# Regional 4x L40S Fixed-DP Comparison

Summarize the May 2024 regional runs for GPT-2 Medium on the local 4x L40S cluster. Each run compares Fixed Start Time Dynamic Parallelism against the best static parallelism configuration for the same region, token budget, sampled offsets, deadline, and microbatch-aware profiler calibration.


In [ ]:
from __future__ import annotations

import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "runs").exists():
    REPO_ROOT = REPO_ROOT.parent

PROFILE_JSON = REPO_ROOT / "calibrations" / "calibration_gpt2_medium_4gpu_l40s_microbatch.json"
TOKEN_SWEEP_RUN_ROOT = REPO_ROOT / "runs" / "region_may2024_gpt2_medium_4gpu_l40s_token_sweep_1b_to_30b_25slack_shared_offsets"
MODEL_SCALING_RUN_ROOT = REPO_ROOT / "runs" / "region_may2024_gpt2_medium_4gpu_l40s_model_scaling_1b_to_30b_25slack_per_run_offsets"
FIG_DIR = REPO_ROOT / "figures" / "experiment_plots" / "regions"
SAVE_FIGURES = True

REGION_ORDER = ["us-west-1", "us-west-2", "us-east-2"]
REGION_LABELS = {
    "us-west-1": "us-west-1",
    "us-west-2": "us-west-2",
    "us-east-2": "us-east-2",
}

MODEL_ORDER = ["gpt2_medium"]
MODEL_LABELS = {"gpt2_medium": "GPT-2 Medium\n4x L40S"}
TOKEN_ORDER = [f"{i}b" for i in range(1, 31)]
TOKEN_VALUES_B = list(range(1, 31))
DP_LABEL = "Fixed Start Time Dynamic Parallelism"

plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
})


def _axes_list(axes):
    return np.ravel(np.atleast_1d(axes)).tolist()


if SAVE_FIGURES:
    FIG_DIR.mkdir(parents=True, exist_ok=True)


## Regional Carbon Trace Shape

Before comparing policy savings, inspect the underlying May 2024 carbon-intensity traces. The first plot uses the same 7-day window for every region so differences in absolute level and short-term variability are visually comparable.

In [ ]:
TRACE_DIR = REPO_ROOT / "data"
TRACE_CHUNK_START = pd.Timestamp("2024-05-10 00:00:00")
TRACE_CHUNK_DAYS = 7

# Figure styling knobs
FIGSIZE = (12.5, 4.8)
LINE_WIDTH = 2.0
AXIS_LABEL_SIZE = 24
TICK_LABEL_SIZE = 24
LEGEND_FONT_SIZE = 20
DATE_ROTATION = 20
FIG_DPI = 300

trace_colors = {
    "us-west-1": "#1f77b4",
    "us-west-2": "#2a9d8f",
    "us-east-2": "#9467bd",
}


def read_region_trace(region: str) -> pd.DataFrame:
    path = TRACE_DIR / f"carbon_2024_05_hourly_aws_{region}.csv"
    if not path.exists():
        raise FileNotFoundError(path)

    trace = pd.read_csv(path)
    trace["timestamp"] = (
        pd.to_datetime(trace[["year", "month", "day"]])
        + pd.to_timedelta(trace["time"])
    )
    trace["region"] = region

    return trace[["region", "timestamp", "carbon"]]


trace_df = pd.concat(
    [read_region_trace(region) for region in REGION_ORDER],
    ignore_index=True,
)

# Original trace is in gCO2e/kWh. Convert to kgCO2e/kWh.
trace_df["carbon_gco2e_per_kwh"] = trace_df["carbon"].astype(float)
trace_df["carbon_kgco2e_per_kwh"] = trace_df["carbon_gco2e_per_kwh"] / 1000.0

trace_chunk_end = TRACE_CHUNK_START + pd.Timedelta(days=TRACE_CHUNK_DAYS)
trace_chunk = trace_df[
    (trace_df["timestamp"] >= TRACE_CHUNK_START)
    & (trace_df["timestamp"] < trace_chunk_end)
].copy()


fig, ax = plt.subplots(figsize=FIGSIZE)

for region in REGION_ORDER:
    chunk_region = trace_chunk[trace_chunk["region"] == region]

    color = trace_colors[region]
    label = REGION_LABELS[region]

    carbon_values = chunk_region["carbon_kgco2e_per_kwh"]
    carbon_cv = carbon_values.std() / carbon_values.mean()

    ax.plot(
        chunk_region["timestamp"],
        carbon_values,
        color=color,
        linewidth=LINE_WIDTH,
        label=f"{label} CV={carbon_cv:.2f}",
    )

ax.set_xlabel("Timestamp", fontsize=AXIS_LABEL_SIZE)
ax.set_ylabel(r"kgCO$_2$e/kWh", fontsize=AXIS_LABEL_SIZE)

ax.tick_params(axis="both", labelsize=TICK_LABEL_SIZE)

ax.grid(True, axis="y", alpha=0.25)
ax.legend(frameon=False, fontsize=LEGEND_FONT_SIZE)

fig.autofmt_xdate(rotation=DATE_ROTATION)
fig.tight_layout()

if SAVE_FIGURES:
    fig.savefig(
        FIG_DIR / "regional_carbon_trace_example_combined.png",
        bbox_inches="tight",
        dpi=FIG_DPI,
    )

plt.show()

## Regional Carbon Variability Metrics

Mean and standard deviation are useful, but normalized spread and percentile ratios better explain how much carbon-arbitrage opportunity each trace offers. Higher coefficient of variation, larger p95/p05 ratios, and larger daily ranges indicate more room for dynamic parallelism to move work away from dirty periods.

In [ ]:
metric_rows = []
for region in REGION_ORDER:
    sub = trace_df[trace_df["region"] == region].sort_values("timestamp")
    ci = sub["carbon"]
    p05, p25, p50, p75, p95 = ci.quantile([0.05, 0.25, 0.50, 0.75, 0.95])
    daily_range = (
        sub.groupby(sub["timestamp"].dt.date)["carbon"]
        .agg(lambda values: values.max() - values.min())
        .mean()
    )
    metric_rows.append({
        "region": region,
        "mean_gco2_per_kwh": ci.mean(),
        "std_gco2_per_kwh": ci.std(ddof=1),
        "cv_pct": 100.0 * ci.std(ddof=1) / ci.mean(),
        "min_gco2_per_kwh": ci.min(),
        "p05_gco2_per_kwh": p05,
        "median_gco2_per_kwh": p50,
        "p95_gco2_per_kwh": p95,
        "max_gco2_per_kwh": ci.max(),
        "p95_minus_p05_gco2_per_kwh": p95 - p05,
        "p95_div_p05": p95 / p05,
        "max_div_min": ci.max() / ci.min(),
        "mean_abs_hourly_delta_gco2_per_kwh": ci.diff().abs().mean(),
        "mean_daily_range_gco2_per_kwh": daily_range,
    })

trace_metrics_df = pd.DataFrame(metric_rows)
trace_metrics_df["region"] = pd.Categorical(
    trace_metrics_df["region"],
    REGION_ORDER,
    ordered=True,
)
trace_metrics_df = trace_metrics_df.sort_values("region").reset_index(drop=True)

display(trace_metrics_df.round(2))

if SAVE_FIGURES:
    trace_metrics_df.to_csv(FIG_DIR / "regional_carbon_variability_metrics.csv", index=False)

fig, axes = plt.subplots(2, 2, figsize=(13.5, 8.0))
axes = axes.ravel()
x = np.arange(len(REGION_ORDER))
bar_colors = [trace_colors[region] for region in REGION_ORDER]

metric_specs = [
    ("mean_gco2_per_kwh", "Mean carbon intensity", "gCO2e/kWh", "std_gco2_per_kwh"),
    ("cv_pct", "Coefficient of variation", "%", None),
    ("p95_div_p05", "p95 / p05 ratio", "ratio", None),
    ("mean_daily_range_gco2_per_kwh", "Mean daily range", "gCO2e/kWh", None),
]

for ax, (column, title, ylabel, error_column) in zip(axes, metric_specs):
    values = trace_metrics_df[column].astype(float).to_numpy()
    yerr = (
        trace_metrics_df[error_column].astype(float).to_numpy()
        if error_column is not None else None
    )
    bars = ax.bar(
        x,
        values,
        yerr=yerr,
        capsize=4 if yerr is not None else 0,
        color=bar_colors,
        edgecolor="#222222",
        linewidth=0.7,
        alpha=0.9,
    )
    for i, (bar, value) in enumerate(zip(bars, values)):
        label_y = bar.get_height()
        if yerr is not None:
            label_y += yerr[i]
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            label_y,
            f"{value:.2f}" if value < 10 else f"{value:.1f}",
            ha="center",
            va="bottom",
            fontsize=9,
        )
    ax.set_title(title)
    ax.set_xticks(x)
    ax.set_xticklabels([REGION_LABELS[r] for r in REGION_ORDER], rotation=20, ha="right")
    ax.set_ylabel(ylabel)
    ax.grid(True, axis="y", alpha=0.25)

fig.suptitle("May 2024 Regional Carbon-Trace Variability", y=1.02)
fig.tight_layout()

if SAVE_FIGURES:
    fig.savefig(FIG_DIR / "regional_carbon_variability_metrics.png", bbox_inches="tight")
plt.show()

<!-- faithful-token-sweep-start -->
## GPT-2 Medium Token Sweep Results

This section visualizes the 90-run shared-offset sweep in `runs/region_may2024_gpt2_medium_4gpu_l40s_token_sweep_1b_to_30b_25slack_shared_offsets`. The runs use the microbatch-aware 4x L40S calibration JSON from `calibrations/calibration_gpt2_medium_4gpu_l40s_microbatch.json`. Primary savings are recomputed from each per-episode `results.csv` against the lowest-carbon deadline-feasible static schedule evaluated for that same `(episode, offset_h)` window. The previous best-single-static-policy metric is retained only as a diagnostic.


In [ ]:
FAITHFUL_SWEEP_ROOT = TOKEN_SWEEP_RUN_ROOT
TOKEN_SWEEP_FIG_DIR = FIG_DIR / "gpt2_medium_4gpu_l40s_token_sweep"
SWEEP_COLORS = {
    "us-west-1": "#1f77b4",
    "us-west-2": "#2a9d8f",
    "us-east-2": "#9467bd",
}

if SAVE_FIGURES:
    TOKEN_SWEEP_FIG_DIR.mkdir(parents=True, exist_ok=True)


def _bool_series(series: pd.Series) -> pd.Series:
    if series.dtype == bool:
        return series
    return series.astype(str).str.lower().isin(["true", "1", "yes"])


def _token_b(token_slug: str) -> int:
    return int(str(token_slug).rstrip("b"))


def _resolve_run_dir(path_value: str) -> Path:
    run_dir = Path(path_value)
    return run_dir if run_dir.is_absolute() else REPO_ROOT / run_dir


def summarize_token_sweep_run(row: pd.Series, *, require_feasible: bool) -> dict:
    run_dir = _resolve_run_dir(row["out_dir"])
    results = pd.read_csv(run_dir / "results.csv")
    results["completed_bool"] = _bool_series(results["completed"])
    results["feasible_bool"] = _bool_series(results["feasible"])
    results["has_reshards"] = results["reshards"].astype(float) > 0

    episode_key_cols = ["episode", "offset_h"]
    static_mask = results["policy"].str.startswith("static(")
    static_candidate_mask = static_mask & results["completed_bool"]
    dp_candidate_mask = (results["policy"] == DP_LABEL) & results["completed_bool"]
    if require_feasible:
        static_candidate_mask = static_candidate_mask & results["feasible_bool"]
        dp_candidate_mask = dp_candidate_mask & results["feasible_bool"]

    static_candidates = results.loc[static_candidate_mask].copy()
    if static_candidates.empty:
        raise ValueError(f"No static candidates found for {run_dir}")

    static_counts = (
        static_candidates.groupby(episode_key_cols, dropna=False)
        .size()
        .rename("feasible_static_candidates_for_episode")
        .reset_index()
    )
    best_static_idx = static_candidates.groupby(episode_key_cols, dropna=False)["carbon_g"].idxmin()
    episode_best_static = (
        static_candidates.loc[best_static_idx, episode_key_cols + ["policy", "carbon_g"]]
        .rename(columns={
            "policy": "episode_best_feasible_static_policy",
            "carbon_g": "episode_best_feasible_static_carbon_g",
        })
        .merge(static_counts, on=episode_key_cols, how="left", validate="one_to_one")
    )
    assert not episode_best_static.duplicated(episode_key_cols).any()

    best_single_static_policy = static_candidates.groupby("policy")["carbon_g"].mean().sort_values().index[0]
    best_single_static_ep = static_candidates.loc[
        static_candidates["policy"] == best_single_static_policy,
        episode_key_cols + ["carbon_g"],
    ].rename(columns={"carbon_g": "best_single_static_policy_carbon_g"})

    dp_ep = results.loc[
        dp_candidate_mask,
        episode_key_cols + ["carbon_g", "feasible_bool", "reshards", "has_reshards"],
    ].rename(columns={"carbon_g": "fixed_dp_carbon_g", "feasible_bool": "fixed_dp_feasible"})

    paired = dp_ep.merge(episode_best_static, on=episode_key_cols, how="inner", validate="one_to_one")
    if paired.empty:
        raise ValueError(f"No paired episodes found for {run_dir}")
    dropped_no_static = len(dp_ep) - len(paired)
    assert (paired["feasible_static_candidates_for_episode"] > 0).all()

    saved = paired["episode_best_feasible_static_carbon_g"] - paired["fixed_dp_carbon_g"]
    savings_pct_by_episode = 100.0 * saved / paired["episode_best_feasible_static_carbon_g"]
    savings_pct = savings_pct_by_episode.mean()
    ci95_pct = (
        1.96 * savings_pct_by_episode.std(ddof=1) / math.sqrt(len(paired))
        if len(paired) > 1 else 0.0
    )
    reshards_gt0 = paired.loc[paired["has_reshards"]]
    savings_pct_reshards_gt0 = (
        (100.0 * (reshards_gt0["episode_best_feasible_static_carbon_g"] - reshards_gt0["fixed_dp_carbon_g"])
         / reshards_gt0["episode_best_feasible_static_carbon_g"]).mean()
        if len(reshards_gt0) else np.nan
    )

    old_paired = paired.merge(best_single_static_ep, on=episode_key_cols, how="inner", validate="one_to_one")
    old_saved = old_paired["best_single_static_policy_carbon_g"] - old_paired["fixed_dp_carbon_g"]
    savings_vs_best_single_static_policy = (
        100.0 * old_saved.mean() / old_paired["best_single_static_policy_carbon_g"].mean()
    )

    completed_dp = results.loc[(results["policy"] == DP_LABEL) & results["completed_bool"]]
    feasible_dp = completed_dp.loc[completed_dp["feasible_bool"]]
    feasible_static = results.loc[static_mask & results["feasible_bool"]]

    token_slug = row["token_slug"]
    return {
        "region": row["region"],
        "model": row["model"],
        "model_label": row["model_label"],
        "token_budget": token_slug,
        "token_b": _token_b(token_slug),
        "tokens": int(row["tokens"]),
        "max_episode_hours": int(row["max_episode_hours"]),
        "max_random_offset_h": float(row["max_random_offset_h"]),
        "shared_offset_window_h": float(row["shared_offset_window_h"]),
        "episode_key_columns": "region,model,token_budget,max_episode_hours,n_gpus,seq_length,episode,offset_h",
        "best_static_policy": "episode_best_feasible_static",
        "best_single_static_policy": best_single_static_policy,
        "best_static_carbon_mean": paired["episode_best_feasible_static_carbon_g"].mean(),
        "fixed_dp_carbon_mean": paired["fixed_dp_carbon_g"].mean(),
        "saved_g_mean": saved.mean(),
        "savings_pct": savings_pct,
        "savings_vs_episode_best_static_pct": savings_pct,
        "savings_vs_episode_best_static_pct_reshards_gt0": savings_pct_reshards_gt0,
        "savings_vs_best_single_static_policy": savings_vs_best_single_static_policy,
        "ci95_pct": ci95_pct,
        "paired_runs": len(paired),
        "best_static_feasible_runs": len(feasible_static),
        "fixed_dp_feasible_runs": len(feasible_dp),
        "fixed_dp_feasible_rate": len(feasible_dp) / max(len(completed_dp), 1),
        "dropped_no_episode_feasible_static_baseline": dropped_no_static,
        "feasible_static_candidates_mean": paired["feasible_static_candidates_for_episode"].mean(),
        "feasible_static_candidates_min": paired["feasible_static_candidates_for_episode"].min(),
        "reshards_zero_runs": int((paired["reshards"] == 0).sum()),
        "reshards_zero_fraction": float((paired["reshards"] == 0).mean()),
        "fixed_dp_reshards_median": paired["reshards"].median(),
        "n_gpus": int(row["n_gpus"]),
        "seq_length": int(row["seq_length"]),
        "strategy_profile_csv": row["strategy_profile_csv"],
        "run_dir": str(run_dir.relative_to(REPO_ROOT)),
    }

faithful_manifest = pd.read_csv(FAITHFUL_SWEEP_ROOT / "manifest.csv")
faithful_feasible_summary = pd.DataFrame([
    summarize_token_sweep_run(row, require_feasible=True)
    for _, row in faithful_manifest.iterrows()
]).sort_values(["region", "model", "token_b"]).reset_index(drop=True)
faithful_completed_summary = pd.DataFrame([
    summarize_token_sweep_run(row, require_feasible=False)
    for _, row in faithful_manifest.iterrows()
]).sort_values(["region", "model", "token_b"]).reset_index(drop=True)

faithful_compare = faithful_feasible_summary.merge(
    faithful_completed_summary[[
        "region", "model", "token_budget", "savings_pct", "saved_g_mean", "paired_runs",
        "savings_vs_best_single_static_policy", "savings_vs_episode_best_static_pct_reshards_gt0",
    ]],
    on=["region", "model", "token_budget"],
    suffixes=("_feasible", "_completed"),
)
faithful_compare["savings_pct_delta_completed_minus_feasible"] = (
    faithful_compare["savings_pct_completed"] - faithful_compare["savings_pct_feasible"]
)

summary_cols = [
    "region", "model", "token_budget", "best_static_policy", "savings_pct",
    "savings_vs_best_single_static_policy", "ci95_pct", "paired_runs",
    "fixed_dp_feasible_runs", "reshards_zero_fraction", "fixed_dp_reshards_median",
]
display(faithful_feasible_summary[summary_cols].round(3))

if SAVE_FIGURES:
    faithful_feasible_summary.to_csv(TOKEN_SWEEP_FIG_DIR / "faithful_token_sweep_feasible_summary.csv", index=False)
    faithful_completed_summary.to_csv(TOKEN_SWEEP_FIG_DIR / "faithful_token_sweep_completed_summary.csv", index=False)
    faithful_compare.to_csv(TOKEN_SWEEP_FIG_DIR / "faithful_token_sweep_feasible_vs_completed.csv", index=False)


### Deadline-Feasible Savings


In [ ]:
# Figure styling knobs
FIGSIZE = (
    (max(10.5, 0.38 * len(TOKEN_VALUES_B) + 4.0), 5.4)
    if len(MODEL_ORDER) == 1 else (13.8, 5.4)
)
LINE_WIDTH = 2.0
MARKER_SIZE = 5
CAPSIZE = 3
AXIS_LABEL_SIZE = 24
TICK_LABEL_SIZE = 24
PANEL_TITLE_SIZE = 20
LEGEND_FONT_SIZE = 20
ANNOTATION_FONT_SIZE = 13
FIG_DPI = 300

fig, axes = plt.subplots(
    1,
    len(MODEL_ORDER),
    figsize=FIGSIZE,
    sharey=True,
)

axes = _axes_list(axes)

for ax, model in zip(axes, MODEL_ORDER):
    sub = faithful_feasible_summary[
        faithful_feasible_summary["model"] == model
    ]

    for region in REGION_ORDER:
        rs = sub[sub["region"] == region].sort_values("token_b")

        ax.errorbar(
            rs["token_b"],
            rs["savings_pct"],
            yerr=rs["ci95_pct"],
            marker="o",
            markersize=MARKER_SIZE,
            linewidth=LINE_WIDTH,
            capsize=CAPSIZE,
            label=REGION_LABELS[region],
            color=SWEEP_COLORS[region],
        )

        # Print/annotate carbon saved percentage above each point.
        for _, row in rs.iterrows():
            x = row["token_b"]
            y = row["savings_pct"]

            if np.isfinite(x) and np.isfinite(y):
                ax.annotate(
                    f"{y:.1f}%",
                    xy=(x, y),
                    xytext=(0, 8),
                    textcoords="offset points",
                    ha="center",
                    va="bottom",
                    fontsize=ANNOTATION_FONT_SIZE,
                    color=SWEEP_COLORS[region],
                )

    ax.axhline(0, color="#333333", linewidth=0.8)
    ax.set_title(MODEL_LABELS[model], fontsize=PANEL_TITLE_SIZE)
    ax.set_xlabel("Training tokens (billions)", fontsize=AXIS_LABEL_SIZE)
    ax.set_xticks(TOKEN_VALUES_B)
    ax.tick_params(axis="both", labelsize=TICK_LABEL_SIZE)
    ax.grid(True, axis="y", alpha=0.25)

Y_LABEL_PAD = 12

axes[0].set_ylabel(
    "Carbon saved (%)",
    fontsize=AXIS_LABEL_SIZE,
    labelpad=Y_LABEL_PAD,
)

axes[-1].legend(
    frameon=False,
    loc="upper right",
    fontsize=LEGEND_FONT_SIZE,
)

fig.tight_layout()

if SAVE_FIGURES:
    fig.savefig(
        TOKEN_SWEEP_FIG_DIR / "faithful_token_sweep_savings_pct.png",
        bbox_inches="tight",
        dpi=FIG_DPI,
    )

plt.show()


In [ ]:
# =====================================================================
# Compact pivot table: rows = model/token budget, columns = regions
# =====================================================================
table = faithful_feasible_summary.copy()

table = table[
    table["model"].isin(MODEL_ORDER)
    & table["region"].isin(REGION_ORDER)
].copy()

table["Model"] = table["model"].map(MODEL_LABELS)
table["Training tokens (B)"] = table["token_b"]
table["Region"] = table["region"].map(REGION_LABELS)
table["Carbon saved"] = table["savings_pct"].map(lambda x: f"{x:.2f}%")

pivot = table.pivot_table(
    index=["Model", "Training tokens (B)"],
    columns="Region",
    values="Carbon saved",
    aggfunc="first",
).reset_index()

print("\nCarbon saved relative to Best Static")
print(pivot.to_string(index=False))

### Absolute Carbon Saved


In [ ]:
fig_width = max(10.5, 0.38 * len(TOKEN_VALUES_B) + 4.0) if len(MODEL_ORDER) == 1 else 13.5
fig, axes = plt.subplots(1, len(MODEL_ORDER), figsize=(fig_width, 4.6), sharey=False)
axes = _axes_list(axes)

for ax, model in zip(axes, MODEL_ORDER):
    sub = faithful_feasible_summary[faithful_feasible_summary["model"] == model]
    for region in REGION_ORDER:
        rs = sub[sub["region"] == region].sort_values("token_b")
        ax.plot(
            rs["token_b"],
            rs["saved_g_mean"],
            marker="o",
            linewidth=2,
            label=REGION_LABELS[region],
            color=SWEEP_COLORS[region],
        )
    ax.axhline(0, color="#333333", linewidth=0.8)
    ax.set_title(MODEL_LABELS[model])
    ax.set_xlabel("Token budget (billions)")
    ax.set_xticks(TOKEN_VALUES_B)
    ax.set_ylabel("Mean carbon saved (gCO2e)")

axes[-1].legend(frameon=False, loc="upper left")
fig.suptitle("Absolute Carbon Saved on Deadline-Feasible Episodes", y=1.02)
fig.tight_layout()

if SAVE_FIGURES:
    fig.savefig(TOKEN_SWEEP_FIG_DIR / "faithful_token_sweep_saved_g.png", bbox_inches="tight")
plt.show()


### Deadline Feasibility


In [ ]:
fig_width = max(10.5, 0.38 * len(TOKEN_VALUES_B) + 4.0) if len(MODEL_ORDER) == 1 else 13.5
fig, axes = plt.subplots(1, len(MODEL_ORDER), figsize=(fig_width, 4.6), sharey=True)
axes = _axes_list(axes)

for ax, model in zip(axes, MODEL_ORDER):
    sub = faithful_feasible_summary[faithful_feasible_summary["model"] == model]
    for region in REGION_ORDER:
        rs = sub[sub["region"] == region].sort_values("token_b")
        ax.plot(
            rs["token_b"],
            100.0 * rs["fixed_dp_feasible_rate"],
            marker="o",
            linewidth=2,
            label=REGION_LABELS[region],
            color=SWEEP_COLORS[region],
        )
    ax.set_title(MODEL_LABELS[model])
    ax.set_xlabel("Token budget (billions)")
    ax.set_xticks(TOKEN_VALUES_B)
    ax.set_ylim(0, 105)
    ax.axhline(100, color="#333333", linewidth=0.8, linestyle="--")

axes[0].set_ylabel("Fixed-DP episodes meeting deadline (%)")
axes[-1].legend(frameon=False, loc="lower left")
fig.suptitle("Deadline Feasibility of Fixed-DP", y=1.02)
fig.tight_layout()

if SAVE_FIGURES:
    fig.savefig(TOKEN_SWEEP_FIG_DIR / "faithful_token_sweep_feasibility_rate.png", bbox_inches="tight")
plt.show()


### Savings Heatmap


In [ ]:
def _pct_label(value: float) -> str:
    if value == 0:
        return "0%"
    if abs(value) < 0.1:
        return f"{value:.2f}%"
    return f"{value:.1f}%"

fig_width = max(10.5, 0.45 * len(TOKEN_VALUES_B) + 4.0) if len(MODEL_ORDER) == 1 else 13.5
fig, axes = plt.subplots(1, len(MODEL_ORDER), figsize=(fig_width, 4.2), sharey=True)
axes = _axes_list(axes)
limit = max(1.0, float(np.nanmax(np.abs(faithful_feasible_summary["savings_pct"]))))

for ax, model in zip(axes, MODEL_ORDER):
    pivot = (
        faithful_feasible_summary[faithful_feasible_summary["model"] == model]
        .pivot(index="region", columns="token_b", values="savings_pct")
        .reindex(index=REGION_ORDER, columns=TOKEN_VALUES_B)
    )
    im = ax.imshow(pivot.values, cmap="RdYlGn", vmin=-limit, vmax=limit, aspect="auto")
    ax.set_title(MODEL_LABELS[model])
    ax.set_xticks(np.arange(len(TOKEN_VALUES_B)))
    ax.set_xticklabels(TOKEN_VALUES_B)
    ax.set_xlabel("Token budget (billions)")
    ax.set_yticks(np.arange(len(REGION_ORDER)))
    ax.set_yticklabels([REGION_LABELS[r] for r in REGION_ORDER])
    ax.grid(False)
    for i, region in enumerate(REGION_ORDER):
        for j, token_b in enumerate(TOKEN_VALUES_B):
            value = pivot.loc[region, token_b]
            ax.text(j, i, _pct_label(value), ha="center", va="center", fontsize=9, color="#111111")

fig.colorbar(im, ax=axes, fraction=0.025, pad=0.02, label="Carbon saved (%)")
fig.suptitle("Deadline-Feasible Savings Heatmap", y=1.02)

if SAVE_FIGURES:
    fig.savefig(TOKEN_SWEEP_FIG_DIR / "faithful_token_sweep_savings_heatmap.png", bbox_inches="tight")
plt.show()


### Feasible vs Completed Diagnostic


In [ ]:
diagnostic_cols = [
    "region", "model", "token_budget", "savings_pct_feasible",
    "fixed_dp_feasible_runs", "feasible_static_candidates_mean",
    "savings_vs_best_single_static_policy_feasible",
    "savings_vs_episode_best_static_pct_reshards_gt0_feasible",
    "reshards_zero_fraction", "dropped_no_episode_feasible_static_baseline",
    "savings_pct_completed", "savings_pct_delta_completed_minus_feasible",
    "paired_runs_feasible", "paired_runs_completed",
]

diagnostic = faithful_compare[diagnostic_cols].sort_values(
    ["model", "token_budget", "region"],
    key=lambda col: col.map(_token_b) if col.name == "token_budget" else col,
)
display(diagnostic.round(3))
# <!-- faithful-token-sweep-end -->


<!-- model-scaling-start -->
## GPT-2 Medium Per-Run Offset Sweep Results

This section visualizes `runs/region_may2024_gpt2_medium_4gpu_l40s_model_scaling_1b_to_30b_25slack_per_run_offsets`, which runs the same regional fixed-start dynamic-parallelism evaluation for GPT-2 Medium on the 4x L40S microbatch-aware calibration with token budgets from 1B through 30B. With only one local hardware profile, this is no longer a cross-model scaling comparison; it keeps the old plotting structure while focusing on the single measured local model.


In [ ]:
MODEL_SCALING_ROOT = MODEL_SCALING_RUN_ROOT
MODEL_SCALING_FIG_DIR = FIG_DIR / "gpt2_medium_4gpu_l40s_model_scaling"

MODEL_SCALING_MODEL_ORDER = MODEL_ORDER
MODEL_SCALING_MODEL_LABELS = MODEL_LABELS
MODEL_SCALING_TOKEN_ORDER = [f"{i}b" for i in range(1, 31)]
MODEL_SCALING_TOKEN_VALUES_B = list(range(1, 31))
MODEL_SCALING_COLORS = {
    "us-west-1": "#1f77b4",
    "us-west-2": "#2a9d8f",
    "us-east-2": "#9467bd",
}

if SAVE_FIGURES:
    MODEL_SCALING_FIG_DIR.mkdir(parents=True, exist_ok=True)


def _token_b_model_scaling(token_slug: str) -> int:
    return int(str(token_slug).rstrip("b"))


def _pct_label_model_scaling(value: float) -> str:
    if pd.isna(value):
        return "n/a"
    return f"{value:.1f}"


model_scaling_summary_path = MODEL_SCALING_ROOT / "combined_fixed_dp_summary.csv"
model_scaling_catalog_path = MODEL_SCALING_ROOT / "model_catalog.csv"
model_scaling_manifest_path = MODEL_SCALING_ROOT / "manifest.csv"
if not model_scaling_summary_path.exists():
    raise FileNotFoundError(model_scaling_summary_path)
if not model_scaling_catalog_path.exists():
    raise FileNotFoundError(model_scaling_catalog_path)
if not model_scaling_manifest_path.exists():
    raise FileNotFoundError(model_scaling_manifest_path)

model_scaling_summary = pd.read_csv(model_scaling_summary_path)
model_scaling_catalog = pd.read_csv(model_scaling_catalog_path)
model_scaling_manifest = pd.read_csv(model_scaling_manifest_path)
manifest_cols = [
    "region", "model", "token_slug", "max_episode_hours",
    "max_random_offset_h", "shared_offset_window_h",
]
model_scaling_manifest_for_summary = model_scaling_manifest[manifest_cols].rename(
    columns={"token_slug": "token_budget"}
)
model_scaling_summary = model_scaling_summary.merge(
    model_scaling_manifest_for_summary,
    on=["region", "model", "token_budget"],
    how="left",
    validate="one_to_one",
)
assert model_scaling_summary["max_random_offset_h"].notna().all()
model_scaling_summary["token_b"] = model_scaling_summary["token_budget"].map(_token_b_model_scaling)
model_scaling_summary["model_label"] = model_scaling_summary["model"].map(MODEL_SCALING_MODEL_LABELS)
model_scaling_summary["region_label"] = model_scaling_summary["region"].map(REGION_LABELS)
model_scaling_summary["savings_pct"] = model_scaling_summary[
    "savings_vs_episode_best_static_pct_mean"
]
model_scaling_summary["savings_pct_reshards_gt0"] = model_scaling_summary[
    "savings_vs_episode_best_static_pct_reshards_gt0_mean"
]
model_scaling_summary["savings_vs_best_single_static_policy_pct"] = model_scaling_summary[
    "savings_vs_best_single_static_policy_pct"
]
model_scaling_summary["true_switch_runs"] = model_scaling_summary["reshards_gt0_runs"].astype(int)
model_scaling_summary["trace_window_status"] = np.where(
    model_scaling_summary["max_random_offset_h"].gt(0),
    "randomized",
    "offset=0",
)

expected_rows = (
    len(REGION_ORDER)
    * len(MODEL_SCALING_MODEL_ORDER)
    * len(MODEL_SCALING_TOKEN_ORDER)
)
assert len(model_scaling_summary) == expected_rows, (
    f"Expected {expected_rows} rows, found {len(model_scaling_summary)}"
)
assert model_scaling_summary["fixed_dp_rows_dropped_no_feasible_static_baseline"].eq(0).all()
assert model_scaling_summary["paired_feasible_runs"].gt(0).all()

model_scaling_summary = model_scaling_summary.sort_values(
    ["region", "model", "token_b"]
).reset_index(drop=True)

catalog_cols = [
    "model", "model_label", "n_gpus", "seq_length", "strategy_count",
    "fastest_throughput_tok_s", "slowest_throughput_tok_s",
]
display(model_scaling_catalog[catalog_cols].round(2))

model_scaling_overview_cols = [
    "region", "model", "token_budget", "max_episode_hours",
    "max_random_offset_h", "trace_window_status", "savings_pct",
    "savings_pct_reshards_gt0", "savings_vs_best_single_static_policy_pct",
    "paired_feasible_runs", "feasible_static_candidates_mean",
    "reshards_zero_fraction", "fixed_dp_reshards_median",
]
display(model_scaling_summary[model_scaling_overview_cols].round(3))

model_scaling_zero_offset = model_scaling_summary[
    model_scaling_summary["max_random_offset_h"].le(0)
][["region", "model", "token_budget", "max_episode_hours", "max_random_offset_h"]]
display(model_scaling_zero_offset.round(3))

if SAVE_FIGURES:
    model_scaling_catalog.to_csv(MODEL_SCALING_FIG_DIR / "model_scaling_catalog.csv", index=False)
    model_scaling_summary.to_csv(MODEL_SCALING_FIG_DIR / "model_scaling_summary.csv", index=False)
    model_scaling_zero_offset.to_csv(MODEL_SCALING_FIG_DIR / "model_scaling_zero_offset_runs.csv", index=False)


### Primary Savings By Model


In [ ]:
n_models = len(MODEL_SCALING_MODEL_ORDER)
n_cols = min(3, n_models)
n_rows = math.ceil(n_models / n_cols)
fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(max(7.4, 4.8 * n_cols, 0.38 * len(MODEL_SCALING_TOKEN_VALUES_B) + 4.0), 4.2 * n_rows),
    sharey=True,
)
axes = _axes_list(axes)

for ax, model in zip(axes, MODEL_SCALING_MODEL_ORDER):
    sub = model_scaling_summary[model_scaling_summary["model"] == model]
    for region in REGION_ORDER:
        rs = sub[sub["region"] == region].sort_values("token_b")
        ax.plot(
            rs["token_b"],
            rs["savings_pct"],
            marker="o",
            linewidth=2,
            label=REGION_LABELS[region],
            color=MODEL_SCALING_COLORS[region],
        )
    ax.axhline(0, color="#333333", linewidth=0.8)
    ax.set_title(MODEL_SCALING_MODEL_LABELS[model])
    ax.set_xlabel("Token budget (billions)")
    ax.set_xticks(MODEL_SCALING_TOKEN_VALUES_B)
    ax.set_ylabel("Carbon saved vs episode-best static (%)")

for ax in axes[n_models:]:
    ax.remove()

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, frameon=False, loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.02))
fig.suptitle("GPT-2 Medium 4x L40S: Deadline-Feasible Fixed-DP Savings", y=1.08)
fig.tight_layout()

if SAVE_FIGURES:
    fig.savefig(MODEL_SCALING_FIG_DIR / "model_scaling_savings_pct.png", bbox_inches="tight")
plt.show()


### 30B-Token Regional Heatmap


In [ ]:
MODEL_SCALING_HEATMAP_TOKEN = "30b"
model_scaling_heatmap = model_scaling_summary[
    model_scaling_summary["token_budget"] == MODEL_SCALING_HEATMAP_TOKEN
]
pivot = (
    model_scaling_heatmap
    .pivot(index="region", columns="model", values="savings_pct")
    .reindex(index=REGION_ORDER, columns=MODEL_SCALING_MODEL_ORDER)
)
limit = max(1.0, float(np.nanmax(np.abs(pivot.values))))

fig_width = max(4.8, 2.2 * len(MODEL_SCALING_MODEL_ORDER) + 3.2)
fig, ax = plt.subplots(figsize=(fig_width, 3.6))
im = ax.imshow(pivot.values, cmap="RdYlGn", vmin=-limit, vmax=limit, aspect="auto")
ax.set_xticks(np.arange(len(MODEL_SCALING_MODEL_ORDER)))
ax.set_xticklabels([MODEL_SCALING_MODEL_LABELS[m] for m in MODEL_SCALING_MODEL_ORDER], rotation=20, ha="right")
ax.set_yticks(np.arange(len(REGION_ORDER)))
ax.set_yticklabels([REGION_LABELS[r] for r in REGION_ORDER])
ax.grid(False)

for i, region in enumerate(REGION_ORDER):
    for j, model in enumerate(MODEL_SCALING_MODEL_ORDER):
        value = pivot.loc[region, model]
        ax.text(j, i, _pct_label_model_scaling(value), ha="center", va="center", fontsize=9, color="#111111")

fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02, label="Carbon saved (%)")
ax.set_title("30B Tokens: Savings vs Episode-Best Feasible Static")
fig.tight_layout()

if SAVE_FIGURES:
    fig.savefig(MODEL_SCALING_FIG_DIR / "model_scaling_30b_savings_heatmap.png", bbox_inches="tight")
plt.show()


### Reshard Diagnostics


In [ ]:
fig, axes = plt.subplots(1, len(REGION_ORDER), figsize=(14.5, 4.6), sharey=True)
bar_width = 0.022
token_offsets = np.linspace(
    -bar_width * (len(MODEL_SCALING_TOKEN_ORDER) - 1) / 2,
    bar_width * (len(MODEL_SCALING_TOKEN_ORDER) - 1) / 2,
    len(MODEL_SCALING_TOKEN_ORDER),
)
x = np.arange(len(MODEL_SCALING_MODEL_ORDER))

for ax, region in zip(axes, REGION_ORDER):
    sub = model_scaling_summary[model_scaling_summary["region"] == region]
    for offset, token in zip(token_offsets, MODEL_SCALING_TOKEN_ORDER):
        ts = (
            sub[sub["token_budget"] == token]
            .set_index("model")
            .reindex(MODEL_SCALING_MODEL_ORDER)
        )
        ax.bar(
            x + offset,
            100.0 * ts["reshards_zero_fraction"],
            width=bar_width,
            label=token,
        )
    ax.set_title(REGION_LABELS[region])
    ax.set_xticks(x)
    ax.set_xticklabels([MODEL_SCALING_MODEL_LABELS[m] for m in MODEL_SCALING_MODEL_ORDER], rotation=35, ha="right")
    ax.set_ylim(0, 105)
    ax.axhline(100, color="#333333", linewidth=0.8, linestyle="--")

axes[0].set_ylabel("Fixed-DP runs with reshards = 0 (%)")
handles, labels = axes[-1].get_legend_handles_labels()
fig.legend(handles, labels, frameon=False, title="Tokens", loc="lower center", ncol=10, bbox_to_anchor=(0.5, -0.05))
fig.suptitle("Model Scaling: No-Switch Fixed-DP Fraction", y=1.02)
fig.tight_layout(rect=(0, 0.08, 1, 1))

if SAVE_FIGURES:
    fig.savefig(MODEL_SCALING_FIG_DIR / "model_scaling_reshards_zero_fraction.png", bbox_inches="tight")
plt.show()

model_scaling_diag_cols = [
    "region", "model", "token_budget", "max_episode_hours",
    "max_random_offset_h", "trace_window_status", "paired_feasible_runs",
    "feasible_static_candidates_mean", "savings_pct",
    "savings_pct_reshards_gt0", "savings_vs_best_single_static_policy_pct",
    "true_switch_runs", "reshards_zero_runs", "reshards_zero_fraction",
    "fixed_dp_reshards_median",
]
model_scaling_diag = model_scaling_summary[model_scaling_diag_cols].copy()
model_scaling_diag = model_scaling_diag.sort_values(
    ["token_budget", "region", "model"],
    key=lambda col: col.map(_token_b_model_scaling) if col.name == "token_budget" else col,
)
display(model_scaling_diag.round(3))

if SAVE_FIGURES:
    model_scaling_diag.to_csv(MODEL_SCALING_FIG_DIR / "model_scaling_reshards_diagnostics.csv", index=False)
# <!-- model-scaling-end -->
